# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by their `@id` fields per the Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema and accessible at:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
We load dataset metadata from the Croissant schema using `mlcroissant`. This allows exploration of all record sets, fields, and metadata via their respective `@id`s.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print overview of the dataset
print('Dataset Title:', dataset.metadata.name)
print('Description:', dataset.metadata.description)
print('Published:', getattr(dataset.metadata, 'datePublished', None))
print('Version:', getattr(dataset.metadata, 'version', None))
print('Identifier:', getattr(dataset.metadata, 'identifier', None))

## 2. Data Overview
Let's list all available **record sets** and their fields, referencing everything by `@id` as required by the Croissant specification.

In [ ]:
# List all record sets and field IDs, referencing everything by @id
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    print('No explicit record sets found in the metadata; attempting to locate schema-defined record sets in the dataset object...')

    # Try inferring record sets using the dataset's ._metadata_json field
    # NOTE: This relies on internal fields, so adjust as appropriate if mlcroissant has new APIs
    meta_json = dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else None
    if meta_json and 'recordSet' in meta_json:
        record_sets = meta_json['recordSet']

# If record_sets are dicts or just @ids, standardize to a flat list of @id strings
record_set_ids = []
if isinstance(record_sets, list):
    for r in record_sets:
        if isinstance(r, dict) and '@id' in r:
            record_set_ids.append(r['@id'])
        elif isinstance(r, str):
            record_set_ids.append(r)
        else:
            pass  # unknown type, skip
elif isinstance(record_sets, dict) and '@id' in record_sets:
    record_set_ids.append(record_sets['@id'])
elif isinstance(record_sets, str):
    record_set_ids.append(record_sets)
else:
    pass

if not record_set_ids:
    # As a fallback, try to parse the record sets by heuristics present in the dataset
    # `dataset.record_sets` often lists RecordSet objects
    if hasattr(dataset, 'record_sets'):
        for rs in dataset.record_sets:
            record_set_ids.append(getattr(rs, '@id', getattr(rs, '_id', None)))

# Remove any None
record_set_ids = [r for r in record_set_ids if r is not None]

if not record_set_ids:
    print('Could not automatically identify record sets by @id. You may need to inspect metadata manually.')
else:
    print(f"Record Sets (@id): {record_set_ids}")

# For each record set, list all fields and their @id
for record_set_id in record_set_ids:
    print(f"\nRecord set @id: {record_set_id}")
    schema = dataset.get_record_set(record_set_id)
    # The 'fields' can also be called 'field' (Croissant spec)
    fields = []
    if hasattr(schema, 'field'):
        fields = schema.field
    elif hasattr(schema, 'fields'):
        fields = schema.fields
    if not fields and isinstance(schema, dict):
        # Try from loaded dict
        fields = schema.get('field', []) or schema.get('fields', [])
    
    field_ids = []
    for f in fields:
        if hasattr(f, '@id'):
            field_ids.append(f'@id: {f.@id} -- schema:name: {getattr(f, "name", "")}')
        elif isinstance(f, dict):
            # dict structure
            f_id = f.get('@id', '<no id>')
            f_name = f.get('name', '')
            field_ids.append(f'@id: {f_id} -- name: {f_name}')
        else:
            field_ids.append(str(f))
    
    print("Fields:")
    for fid in field_ids:
        print('  ', fid)

## 3. Data Extraction
We will load one or more record sets (by their respective `@id`) into Pandas DataFrames for analysis, referencing only by `@id`. If multiple record sets exist, all are showcased.

In [ ]:
# Extract data from all record sets using their @id
if not record_set_ids:
    raise RuntimeError('No record set ids found in metadata, cannot proceed.')

dataframes = {}

for record_set_id in record_set_ids:
    print(f'\nLoading records from @id: {record_set_id}')
    # The records generator returns a sequence of dicts mapping field @id to value
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f'Loaded DataFrame for record set: {record_set_id}, shape={df.shape}')
    print('Columns:', df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Now, let's conduct simple data processing steps on a selected record set (by `@id`):
- Filtering by a numeric field
- Normalization (z-score) of that field
- Grouping by a categorical field

In [ ]:
# Select one record set to explore
# (If more than one is present, simply take the first as an example)
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Print all column (field) @id for exploration
print('Columns in selected DataFrame:', df.columns.tolist())

# Find a likely numeric field and a group/categorical field
numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if not numeric_candidates:
    # Try to infer numeric columns - e.g., try to coerce each column to numeric
    for col in df.columns:
        try:
            _ = pd.to_numeric(df[col].dropna().values[:10])
            numeric_candidates.append(col)
        except Exception:
            continue

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f'Selected numeric field @id: {numeric_field_id}')
else:
    print('No obvious numeric field found.')
    numeric_field_id = None

# For group field, pick a non-numeric field
group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].dtype == object:
        group_field_id = col
        break
if group_field_id:
    print(f'Selected group (categorical) field @id: {group_field_id}')

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Coerce numeric field to float
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    print(f"\nFiltering rows where {numeric_field_id} > 10...")
    filtered_df = df[df[numeric_field_id] > 10].copy()
    print(f"Filtered records: {len(filtered_df)}")

    filtered_df[f'{numeric_field_id}_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nFirst few rows after normalization:")
    display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
        print(f'\nGrouped data by {group_field_id} showing mean {numeric_field_id}')
        display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and its relationship with the group field (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of field @id: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        # Boxplot of numeric field by group
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to:
- Load mass spectrometry data from a Croissant schema (FAIR^2)
- Reference all entities by their Croissant `@id`
- Review available record sets and fields
- Extract data records and load into DataFrames
- Perform basic EDA (filtering, normalization, and grouping)
- Visualize distributions

To go further, explore the full schema to identify more advanced relationships, clean or join multiple record sets, or apply domain-specific statistical analyses. All code operates by referencing schema entities via their `@id`—ensuring robust schema-first data science on complex open datasets.